In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, current_date, datediff, col

# Start Spark session
spark = SparkSession.builder \
    .appName("SupplyChainDelayProcessing") \
    .getOrCreate()

# Step 1: Load CSV data
df = spark.read.option("header", True).csv("/Volumes/workspace/default/subramani/supply_chain_orders.csv")
print("🔹 Raw Data:")
df.show()

# Step 2: Convert delivery_date to proper date format
df = df.withColumn("delivery_date", to_date(col("delivery_date"), "yyyy-MM-dd"))
print("🔹 After converting delivery_date to date:")
df.show()

# Step 3: Calculate delay_days
df = df.withColumn("delay_days", datediff(current_date(), col("delivery_date")))
print("🔹 After calculating delay_days:")
df.show()

# Step 4: Filter delayed shipments (delay_days > 0)
delayed_df = df.filter(col("delay_days") > 0)
print("🔹 Filtered delayed shipments (delay_days > 0):")
delayed_df.show()

# Step 5: Group by supplier_id and count delayed shipments
summary_df = delayed_df.groupBy("supplier_id") \
    .count() \
    .withColumnRenamed("count", "delayed_shipments")
print("🔹 Grouped delay summary by supplier:")
summary_df.show()

# Step 6: Save result to workspace path (no overwrite of previous files)
output_path = "/Volumes/workspace/default/subramani/output_temp"
summary_df.coalesce(1).write.mode("overwrite").option("header", True).csv(output_path)

print("✅ Output saved to folder:", output_path)

# Stop Spark session
spark.stop()


🔹 Raw Data:
+--------+-----------+-------------+----------+--------+
|order_id|supplier_id|delivery_date|   product|quantity|
+--------+-----------+-------------+----------+--------+
|   O1001|       S001|   2024-06-20|    Laptop|      10|
|   O1002|       S002|   2024-06-15|  Keyboard|      50|
|   O1003|       S001|   2024-06-30|     Mouse|      25|
|   O1004|       S003|   2024-05-20|   Monitor|      15|
|   O1005|       S002|   2024-06-10| USB Cable|     100|
|   O1006|       S001|   2024-06-25|   Printer|       5|
|   O1007|       S003|   2024-04-28|Desk Chair|       8|
|   O1008|       S004|   2024-06-01|    Router|      12|
+--------+-----------+-------------+----------+--------+

🔹 After converting delivery_date to date:
+--------+-----------+-------------+----------+--------+
|order_id|supplier_id|delivery_date|   product|quantity|
+--------+-----------+-------------+----------+--------+
|   O1001|       S001|   2024-06-20|    Laptop|      10|
|   O1002|       S002|   2024-06-